In [1]:
%pip install -U langchain langchain-core langchain-google-genai langchain-community \
    langchain-text-splitters langchain-chroma langchain-huggingface \
    sentence_transformers chromadb pypdf tavily-python "requests==2.32.4"

  Using cached langchain-1.2.12-py3-none-any.whl.metadata (5.7 kB)
  Using cached langchain_core-1.2.19-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_google_genai-4.2.1-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached langchain_huggingface-1.2.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_classic-1.0.3-py3-none-any.whl.metadata (4.8 kB)
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_community-0.4-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.31-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_community-0.3.30-py3-none-any.whl.metadata (3.0 kB)
  Using cached l

In [2]:
import json
import os
import shutil
from pathlib import Path
from typing import Any, Optional

import requests
from tavily import TavilyClient

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ── Secret loader (works for both env vars and Colab Secrets) ──────────────
def load_secret(name: str) -> Optional[str]:
    value = os.getenv(name)
    if value:
        return value
    try:
        from google.colab import userdata
        try:
            value = userdata.get(name)
            if value:
                os.environ[name] = value
                return value
        except Exception:
            return None
    except Exception:
        return None
    return None


def load_keys() -> tuple[str, str]:
    google_key = load_secret("GOOGLE_API_KEY") or load_secret("GEMINI_API_KEY")
    tavily_key = load_secret("TAVILY_API_KEY")
    if not google_key:
        raise RuntimeError("Gemini API key not found. Add GOOGLE_API_KEY in Colab Secrets.")
    if not tavily_key:
        raise RuntimeError("Tavily API key not found. Add TAVILY_API_KEY in Colab Secrets.")
    os.environ["GOOGLE_API_KEY"] = google_key
    os.environ["GEMINI_API_KEY"] = google_key
    os.environ["TAVILY_API_KEY"] = tavily_key
    return google_key, tavily_key


# ── Normalize AIMessage to plain text ──────────────────────────────────────
def message_to_text(message) -> str:
    text_attr = getattr(message, "text", None)
    if isinstance(text_attr, str) and text_attr.strip():
        return text_attr.strip()
    content = getattr(message, "content", None)
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                text = item.get("text")
                if text:
                    parts.append(text)
        return "\n".join(parts).strip()
    return ""


GOOGLE_API_KEY, TAVILY_API_KEY = load_keys()
MODEL_NAME = "gemini-2.5-flash"

llm = ChatGoogleGenerativeAI(model=MODEL_NAME, temperature=0)
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

print(f"✅ Keys loaded. Model: {MODEL_NAME}")

✅ Keys loaded. Model: gemini-2.5-flash


Upload PDFs & Build RAG Vector Store

In [3]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
UPLOAD_DIR = Path("/content/student_assistant_uploads")
CHROMA_DIR = Path("/content/student_assistant_chroma")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# ── Upload or reuse existing PDFs ─────────────────────────────────────────
existing_pdfs = sorted(UPLOAD_DIR.glob("*.pdf"))
if existing_pdfs:
    print("Found existing PDFs:")
    for p in existing_pdfs:
        print(" -", p.name)
else:
    from google.colab import files
    print("Please upload one or more PDF files:\n")
    uploaded = files.upload()
    for filename, file_bytes in uploaded.items():
        save_path = UPLOAD_DIR / filename
        with open(save_path, "wb") as f:
            f.write(file_bytes)

pdf_paths = sorted(UPLOAD_DIR.glob("*.pdf"))
if not pdf_paths:
    raise RuntimeError("No PDFs found. Upload at least one PDF to continue.")

# ── Load pages ────────────────────────────────────────────────────────────
documents = []
for path in pdf_paths:
    loader = PyPDFLoader(str(path))
    documents.extend(loader.load())

print(f"\nPages loaded: {len(documents)}")

# ── Chunk ─────────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

print(f"Chunks created: {len(chunks)}")

# ── Embed & store in Chroma ───────────────────────────────────────────────
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="student_assistant_rag",
    persist_directory=str(CHROMA_DIR),
)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print(f"✅ RAG vector store ready. Chunks indexed: {len(chunks)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Please upload one or more PDF files:



Saving COMPUTER-NETORK-LAB-MANUAL.pdf to COMPUTER-NETORK-LAB-MANUAL.pdf

Pages loaded: 52
Chunks created: 82
✅ RAG vector store ready. Chunks indexed: 82


Define All 4 Tools

In [4]:
# ── Helper: format retrieved chunks ──────────────────────────────────────
def format_docs(docs) -> str:
    formatted = []
    for idx, doc in enumerate(docs, start=1):
        source = Path(doc.metadata.get("source", "unknown")).name
        page = doc.metadata.get("page")
        page_label = f"p.{page + 1}" if isinstance(page, int) else "p.?"
        snippet = " ".join(doc.page_content.split())
        formatted.append(f"[Chunk {idx} | {source} | {page_label}]\n{snippet}")
    return "\n\n".join(formatted)


# ── Tool 1: Weather (Open-Meteo, 100% free, no API key) ───────────────────
WEATHER_CODE_LABELS = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog", 51: "Light drizzle",
    53: "Moderate drizzle", 55: "Dense drizzle", 61: "Slight rain",
    63: "Moderate rain", 65: "Heavy rain", 71: "Slight snow",
    73: "Moderate snow", 75: "Heavy snow", 80: "Rain showers",
    81: "Moderate rain showers", 82: "Violent rain showers", 95: "Thunderstorm",
}


def geocode_location(location: str) -> dict:
    response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": location, "count": 1, "language": "en", "format": "json"},
        timeout=30,
    )
    response.raise_for_status()
    results = response.json().get("results") or []
    if not results:
        raise ValueError(f"Could not resolve location: {location}")
    return results[0]


@tool
def get_current_weather(location: str) -> str:
    """Get the current weather for a city or place name."""
    place = geocode_location(location)
    lat, lon = place["latitude"], place["longitude"]

    response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,apparent_temperature,relative_humidity_2m,wind_speed_10m,weather_code",
            "timezone": "auto",
        },
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    current = data.get("current", {})
    units = data.get("current_units", {})

    location_label = ", ".join(
        p for p in [place.get("name", location), place.get("admin1", ""), place.get("country", "")] if p
    )
    weather_label = WEATHER_CODE_LABELS.get(current.get("weather_code"), "Unknown")

    return "\n".join([
        f"Location: {location_label}",
        f"Condition: {weather_label}",
        f"Temperature: {current.get('temperature_2m')} {units.get('temperature_2m', '')}",
        f"Feels like: {current.get('apparent_temperature')} {units.get('apparent_temperature', '')}",
        f"Humidity: {current.get('relative_humidity_2m')} {units.get('relative_humidity_2m', '')}",
        f"Wind speed: {current.get('wind_speed_10m')} {units.get('wind_speed_10m', '')}",
        f"Observed at: {current.get('time')}",
    ])


# ── Tool 2: Web Search (Tavily) ───────────────────────────────────────────
@tool
def web_search(query: str) -> dict:
    """Search the web for fresh or external information."""
    q_lower = query.lower()
    freshness_words = ["latest", "recent", "today", "this week", "news", "update", "release", "deadline"]
    topic = "news" if any(w in q_lower for w in freshness_words) else "general"

    response = tavily_client.search(
        query=query,
        topic=topic,
        max_results=3,
        include_answer=False,
        include_raw_content=False,
        timeout=15,
    )
    results = [
        {"title": r.get("title", ""), "url": r.get("url", ""), "content": r.get("content", "")}
        for r in response.get("results", [])[:3]
    ]
    return {"query": query, "results": results}


print("✅ All tools defined.")

✅ All tools defined.


Router Logic

In [5]:
ROUTE_SYSTEM_PROMPT = """You are a query router for a student assistant system.
Classify the user query into EXACTLY one of these four routes:

- "weather"    → query is about weather, temperature, rain, forecast, humidity, umbrella
- "web_search" → query needs current/recent info, news, live data, updates, events
- "rag"        → query is about uploaded documents, PDFs, course notes, study material
- "direct"     → general knowledge, concepts, definitions, theory, explanations

Reply with ONLY one word. No punctuation. No explanation. Just one of: weather, web_search, rag, direct"""


def classify_route(query: str) -> str:
    """Use LLM to classify the query into one of four routes."""
    messages = [
        SystemMessage(content=ROUTE_SYSTEM_PROMPT),
        HumanMessage(content=query),
    ]
    response = llm.invoke(messages)
    route = message_to_text(response).strip().lower().strip(".")

    valid_routes = {"weather", "web_search", "rag", "direct"}
    if route not in valid_routes:
        # Keyword fallback if LLM gives unexpected output
        q = query.lower()
        if any(w in q for w in ["weather", "temperature", "rain", "forecast", "umbrella", "sunny", "humid", "wind"]):
            return "weather"
        if any(w in q for w in ["latest", "news", "recent", "today", "update", "current", "2024", "2025"]):
            return "web_search"
        if any(w in q for w in ["document", "pdf", "uploaded", "notes", "file", "paper", "summarize the"]):
            return "rag"
        return "direct"

    return route


print("✅ Router defined.")

✅ Router defined.


 Four Handlers + Main Assistant Function

In [6]:
# ── Handler 1: Weather ────────────────────────────────────────────────────
def handle_weather(query: str) -> dict[str, Any]:
    weather_llm = llm.bind_tools([get_current_weather])
    messages: list[BaseMessage] = [
        SystemMessage(content="You are a weather assistant. Always use get_current_weather to fetch live data. Then give a clear, friendly answer."),
        HumanMessage(content=query),
    ]
    ai_msg = weather_llm.invoke(messages)
    messages.append(ai_msg)

    for tool_call in (ai_msg.tool_calls or []):
        tool_result = get_current_weather.invoke(tool_call)
        messages.append(tool_result)

    final = weather_llm.invoke(messages)
    return {"route": "weather", "answer": message_to_text(final)}


# ── Handler 2: Web Search ─────────────────────────────────────────────────
def handle_web_search(query: str) -> dict[str, Any]:
    search_llm = llm.bind_tools([web_search])
    messages: list[BaseMessage] = [
        SystemMessage(content="You are a research assistant. Use web_search to find current information. Call it once with a precise query."),
        HumanMessage(content=query),
    ]
    ai_msg = search_llm.invoke(messages)

    if not ai_msg.tool_calls:
        return {"route": "web_search", "answer": message_to_text(ai_msg)}

    tool_call = ai_msg.tool_calls[0]
    tool_result = web_search.invoke(tool_call)

    raw = getattr(tool_result, "content", "")
    if not isinstance(raw, str):
        raw = str(raw)

    try:
        payload = json.loads(raw)
        result_lines = []
        for idx, item in enumerate(payload.get("results", []), start=1):
            result_lines.append(
                f"{idx}. {item.get('title', '')}\nURL: {item.get('url', '')}\nSnippet: {item.get('content', '')}"
            )
        context = "\n\n".join(result_lines) if result_lines else "No results found."
    except Exception:
        context = raw or "No results found."

    final_messages: list[BaseMessage] = [
        SystemMessage(content="Answer using only the provided search results. Be concise and accurate. Mention sources."),
        HumanMessage(content=f"Question:\n{query}\n\nSearch results:\n{context}"),
    ]
    final = llm.invoke(final_messages)
    return {"route": "web_search", "answer": message_to_text(final)}


# ── Handler 3: RAG ────────────────────────────────────────────────────────
def handle_rag(query: str) -> dict[str, Any]:
    docs = retriever.invoke(query)
    context = format_docs(docs)

    prompt = f"""You are helping students answer questions from their uploaded documents.
Use only the retrieved context below to answer.
If the context is insufficient, say that clearly instead of guessing.

Question:
{query}

Retrieved context:
{context}""".strip()

    response = llm.invoke(prompt)

    sources = []
    for doc in docs:
        source = Path(doc.metadata.get("source", "unknown")).name
        page = doc.metadata.get("page")
        page_label = f"p.{page + 1}" if isinstance(page, int) else "p.?"
        entry = f"{source} | {page_label}"
        if entry not in sources:
            sources.append(entry)

    return {
        "route": "rag",
        "answer": message_to_text(response),
        "sources": sources,
    }


# ── Handler 4: Direct ─────────────────────────────────────────────────────
def handle_direct(query: str) -> dict[str, Any]:
    messages: list[BaseMessage] = [
        SystemMessage(content="You are a knowledgeable and helpful student assistant. Answer clearly and concisely."),
        HumanMessage(content=query),
    ]
    response = llm.invoke(messages)
    return {"route": "direct", "answer": message_to_text(response)}


# ── Main Student Assistant ────────────────────────────────────────────────
def student_assistant(query: str) -> dict[str, Any]:
    """
    Routes the query to one of four handlers:
      weather    → Open-Meteo live weather
      web_search → Tavily search + LLM synthesis
      rag        → ChromaDB retrieval + LLM grounded answer
      direct     → LLM general knowledge answer
    """
    print(f"\n{'='*70}")
    print(f"🧑 Query   : {query}")

    route = classify_route(query)
    print(f"🔀 Route   : {route.upper()}")
    print("-" * 70)

    if route == "weather":
        result = handle_weather(query)
    elif route == "web_search":
        result = handle_web_search(query)
    elif route == "rag":
        result = handle_rag(query)
    else:
        result = handle_direct(query)

    print(f"💬 Answer  :\n{result['answer']}")

    if result.get("sources"):
        print(f"\n📄 Sources : {result['sources']}")

    print("=" * 70)
    return result


print("✅ Student Assistant is ready!")

✅ Student Assistant is ready!


Testing

In [7]:
test_queries = [
    # Direct
    "What is the difference between supervised and unsupervised learning?",
    # Weather
    "What is the weather in Kolkata right now? Should I carry an umbrella?",
    # Web Search
    "What are the latest AI research updates in 2025?",
    # RAG
    "Summarize the main topics from the uploaded documents.",
]

all_results = []
for query in test_queries:
    result = student_assistant(query)
    all_results.append(result)


🧑 Query   : What is the difference between supervised and unsupervised learning?
🔀 Route   : DIRECT
----------------------------------------------------------------------
💬 Answer  :
The main difference between supervised and unsupervised learning lies in the **type of data they use** and their **objective**:

1.  **Supervised Learning:**
    *   **Data:** Uses **labeled data**, meaning each input example has a corresponding correct output or "label."
    *   **Objective:** To learn a mapping from inputs to outputs so it can predict the output for new, unseen inputs. It's like learning with a teacher who provides the correct answers.
    *   **Examples:** Classification (e.g., spam detection, image recognition) and Regression (e.g., predicting house prices, stock prices).

2.  **Unsupervised Learning:**
    *   **Data:** Uses **unlabeled data**, meaning there are no predefined output labels or correct answers.
    *   **Objective:** To find hidden patterns, structures, or relationship

* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 25.820397858s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 5
}
, retry_delay {
  seconds: 25
}
].
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 23.576901832s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com

KeyboardInterrupt: 